# **Нейросети для CATE: TARNet, CFRNet и DragonNet**

Практика к модулю [«Гетерогенные эффекты»](https://ai-interpretability.school), урок «Нейросети для CATE».

В уроке три архитектуры были схемами. TARNet ([Shalit, Johansson и Sontag, 2017](https://arxiv.org/abs/1606.03976)) учит общее представление $\Phi(x)$ и две головы исхода, по одной на группу. CFRNet добавляет к потерям штраф за различие распределений представления в двух группах. DragonNet ([Shi, Blei и Veitch, 2019](https://arxiv.org/abs/1906.02120)) добавляет третью голову — склонность — и targeted regularization, поправку, после которой оценка среднего эффекта удовлетворяет оценивающему уравнению дважды робастной оценки. Здесь мы соберем все три на `torch` и проверим утверждения урока:

1. общая часть сети дает оценку эффекта лучше, чем две отдельные модели T-learner;
2. штраф CFRNet сближает представления групп, а на ошибку оценки эффекта влияет слабее;
3. targeted regularization уменьшает ошибку среднего эффекта, а не CATE;
4. одна сеть с одним сидом — не результат: разброс между сидами того же порядка, что разница между архитектурами.

План тетради:

1. данные IHDP — те же, что в тетради «R-learner»;
2. общая сеть и обучение с ранней остановкой; TARNet;
3. CFRNet: штраф за различие представлений;
4. DragonNet и targeted regularization;
5. где ломается: обучение без ранней остановки и разброс по сидам;
6. реализации.

Тетрадь идет на процессоре; весь прогон занимает три-шесть минут: 35 небольших сетей, каждая учится несколько секунд.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

SEED = 0
torch.set_num_threads(1)     # один поток: одинаковый порядок сложений при каждом запуске
pd.set_option('display.width', 140)

## 1. Данные IHDP

Полусинтетический набор [Hill (2011)](https://doi.org/10.1198/jcgs.2010.08162) на признаках настоящего эксперимента IHDP: 747 детей, 139 в группе воздействия, 25 признаков; исходы сгенерированы по известной формуле, поэтому у каждого ребенка известен истинный эффект $\tau(x)=\mu_1(x)-\mu_0(x)$. Берем первую реализацию из [репозитория CEVAE](https://github.com/AMLab-Amsterdam/CEVAE) (Louizos et al., 2017) и то же разбиение 70/30, что в тетради «R-learner»: там на этой отложенной части S-learner дал PEHE 0.459, T-learner на градиентном бустинге — 0.859.

Ошибку меряем, как в тетради «R-learner»: $\mathrm{PEHE}=\sqrt{\tfrac1n\sum_i(\hat\tau(x_i)-\tau(x_i))^2}$ на отложенной части (у Shalit et al. это $\sqrt{\epsilon_{\mathrm{PEHE}}}$), и ошибку среднего эффекта $|\bar{\hat\tau}-\bar\tau|$ на той же части. Для ранней остановки из обучающей части откладываем еще 20% — проверочную часть, на которой следим за ошибкой прогноза наблюдаемого исхода: истинного эффекта сеть во время обучения не видит.

In [ ]:
URL = 'https://raw.githubusercontent.com/AMLab-Amsterdam/CEVAE/master/datasets/IHDP/csv/ihdp_npci_1.csv'
cols = ['treatment', 'y_factual', 'y_cfactual', 'mu0', 'mu1'] + [f'x{i}' for i in range(1, 26)]
df = pd.read_csv(URL, header=None, names=cols)
X = df[[f'x{i}' for i in range(1, 26)]].values
T = df.treatment.values.astype(int)
Y = df.y_factual.values
tau = (df.mu1 - df.mu0).values

tr, te = train_test_split(np.arange(len(df)), test_size=0.3, random_state=SEED, stratify=T)
fit_idx, val_idx = train_test_split(tr, test_size=0.2, random_state=SEED, stratify=T[tr])
print('обучение:', len(fit_idx), '| проверка для ранней остановки:', len(val_idx), '| отложенная часть:', len(te))
print('доля группы воздействия в обучении:', round(T[fit_idx].mean(), 3))

## 2. Общая сеть и TARNet

Одна сеть на все три архитектуры:

- представление $\Phi(x)$: два слоя по 100 нейронов с активацией ELU, затем нормировка строки на единичную длину — Shalit et al. нормируют $\Phi$, потому что штраф CFRNet иначе зависит от масштаба представления;
- две головы исхода $h_0(\Phi)$ и $h_1(\Phi)$: по слою из 50 нейронов и выходу;
- голова склонности $g(\Phi)=\sigma(w^\top\Phi+b)$ — линейная, как у Shi et al.; у TARNet и CFRNet она не используется;
- параметр $\varepsilon$ для targeted regularization.

Обучение — полным батчем, Adam со скоростью $10^{-3}$ и `weight_decay` $10^{-4}$, до 3000 шагов; запоминаем веса на шаге с наименьшей ошибкой прогноза наблюдаемого исхода на проверочной части. Исход стандартизуем по обучающей части и возвращаем масштаб при прогнозе.

Потери TARNet — взвешенная ошибка наблюдаемого исхода: голова $h_{t_i}$ предсказывает $y_i$, а веса $w_i=\frac{t_i}{2u}+\frac{1-t_i}{2(1-u)}$, где $u$ — доля группы воздействия, уравнивают вклад двух групп (Shalit et al., формула (3)):

$$\mathcal L_{\mathrm{TAR}}=\frac1n\sum_i w_i\,\bigl(h_{t_i}(\Phi(x_i))-y_i\bigr)^2.$$

Функция `fit` ниже обучает все три архитектуры; штраф CFRNet `ipm_linear` вы напишете в разделе 3, а оценку эффекта после targeted regularization — в разделе 4.

In [ ]:
class Net(nn.Module):
    def __init__(self, d, h=100, hh=50):
        super().__init__()
        self.phi = nn.Sequential(nn.Linear(d, h), nn.ELU(), nn.Linear(h, h), nn.ELU())
        self.h0 = nn.Sequential(nn.Linear(h, hh), nn.ELU(), nn.Linear(hh, 1))
        self.h1 = nn.Sequential(nn.Linear(h, hh), nn.ELU(), nn.Linear(hh, 1))
        self.g = nn.Linear(h, 1)
        self.eps = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        z = self.phi(x)
        z = z / z.norm(dim=1, keepdim=True)
        return z, self.h0(z).squeeze(1), self.h1(z).squeeze(1), torch.sigmoid(self.g(z)).squeeze(1)


def tensors(idx, ym, ys):
    return (torch.tensor(X[idx], dtype=torch.float32), torch.tensor(T[idx], dtype=torch.float32),
            torch.tensor((Y[idx] - ym) / ys, dtype=torch.float32))


def fit(kind='tarnet', alpha=0.0, beta=0.0, seed=SEED, steps=3000, early_stop=True):
    # kind: 'tarnet' (с alpha > 0 это CFRNet) или 'dragonnet' (с beta > 0 — targeted regularization)
    torch.manual_seed(seed)
    net = Net(X.shape[1])
    ym, ys = Y[fit_idx].mean(), Y[fit_idx].std()
    x, t, y = tensors(fit_idx, ym, ys)
    xv, tv, yv = tensors(val_idx, ym, ys)
    u = t.mean()
    w = t / (2 * u) + (1 - t) / (2 * (1 - u))
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
    best, state = np.inf, None
    for step in range(steps):
        opt.zero_grad()
        z, q0, q1, g = net(x)
        q = t * q1 + (1 - t) * q0
        if kind == 'dragonnet':
            # Shi et al., (2.2): ошибка исхода плюс alpha * кросс-энтропия склонности
            loss = ((q - y) ** 2).mean() + alpha * nn.functional.binary_cross_entropy(g.clamp(1e-3, 1 - 1e-3), t)
            if beta > 0:
                # Shi et al., (3.2): Q~ = Q + eps * (t/g - (1-t)/(1-g))
                gc = g.clamp(0.01, 0.99)
                q_tilde = q + net.eps * (t / gc - (1 - t) / (1 - gc))
                loss = loss + beta * ((y - q_tilde) ** 2).mean()
        else:
            loss = (w * (q - y) ** 2).mean()
            if alpha > 0:
                loss = loss + alpha * ipm_linear(z, t)
        loss.backward()
        opt.step()
        with torch.no_grad():
            _, v0, v1, _ = net(xv)
            val = float(((tv * v1 + (1 - tv) * v0 - yv) ** 2).mean())
        if val < best or not early_stop:
            best, best_step, state = val, step, {k: v.clone() for k, v in net.state_dict().items()}
    net.load_state_dict(state)
    with torch.no_grad():
        z, q0, q1, g = net(torch.tensor(X, dtype=torch.float32))
    # прогнозы для всех 747 детей, в исходном масштабе
    return {'q0': q0.numpy() * ys + ym, 'q1': q1.numpy() * ys + ym, 'g': g.numpy(), 'eps': float(net.eps.detach()) * ys,
            'z': z.numpy(), 'best_step': best_step}


SEEDS = range(5)

**Задание 1. Напишите функцию `errors(cate)`, которая возвращает PEHE и ошибку среднего эффекта на отложенной части `te`. Обучите TARNet (`fit('tarnet')`) с сидами 0–4, оценка эффекта — `q1 - q0`. Выведите PEHE и ошибку среднего эффекта для каждого сида и их средние. Чему равен средний по пяти сидам PEHE TARNet?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 3. CFRNet: штраф за различие представлений

Теорема 1 Shalit et al. ограничивает $\epsilon_{\mathrm{PEHE}}$ сверху суммой ошибок прогноза наблюдаемого исхода в двух группах и расстояния между распределениями представления $\Phi$ у воздействованных и контрольных — интегральной вероятностной метрики (IPM). CFRNet добавляет это расстояние к потерям с весом $\alpha$:

$$\mathcal L_{\mathrm{CFR}}=\mathcal L_{\mathrm{TAR}}+\alpha\cdot\mathrm{IPM}\bigl(\{\Phi(x_i)\}_{t_i=0},\{\Phi(x_i)\}_{t_i=1}\bigr).$$

Авторы берут расстояние Вассерштейна или MMD. Самый простой вариант — квадрат линейного MMD, он же квадрат расстояния между средними представлениями групп (в таблице 1 статьи — «CFR MMD», «squared linear MMD»):

$$\mathrm{MMD}^2_{\mathrm{lin}}=\Bigl\|\frac1{n_1}\sum_{t_i=1}\Phi(x_i)-\frac1{n_0}\sum_{t_i=0}\Phi(x_i)\Bigr\|^2.$$

При $\alpha=0$ получается TARNet.

**Задание 2. Напишите `ipm_linear(z, t)` на `torch`. Обучите CFRNet при $\alpha\in\{1, 10, 100\}$ с сидами 0–4. Для TARNet и каждого $\alpha$ выведите средние по сидам PEHE, ошибку среднего эффекта и линейный MMD$^2$ представления на обучающей части `fit_idx`. Как растущий $\alpha$ меняет расстояние между представлениями и как — PEHE?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 4. DragonNet и targeted regularization

DragonNet обучает голову склонности вместе с головами исхода (Shi et al., формула (2.2)):

$$\hat R(\theta)=\frac1n\sum_i\bigl(Q(t_i,x_i)-y_i\bigr)^2+\alpha\cdot\mathrm{CrossEntropy}\bigl(g(x_i),t_i\bigr),$$

где $Q(t,x)=h_t(\Phi(x))$. Довод авторов: для поправки на конфаундеры достаточно склонности (теорема Розенбаума и Рубина), поэтому представление полезно сжать до той информации, которая предсказывает воздействие. Оценка среднего эффекта — $\hat\psi_Q=\frac1n\sum_i\bigl(Q(1,x_i)-Q(0,x_i)\bigr)$.

Targeted regularization добавляет один параметр $\varepsilon$ и слагаемое с весом $\beta$ (формулы (3.2)–(3.4)):

$$\tilde Q(t,x)=Q(t,x)+\varepsilon\Bigl(\frac{t}{g(x)}-\frac{1-t}{1-g(x)}\Bigr),\qquad \hat R(\theta)+\beta\,\frac1n\sum_i\bigl(y_i-\tilde Q(t_i,x_i)\bigr)^2.$$

Производная по $\varepsilon$ в минимуме равна нулю, и это ровно оценивающее уравнение дважды робастной оценки среднего эффекта с моделями $\tilde Q$ и $g$. Оценка эффекта объекта после поправки — $\tilde Q(1,x)-\tilde Q(0,x)=Q(1,x)-Q(0,x)+\varepsilon\bigl(\frac1{g(x)}+\frac1{1-g(x)}\bigr)$. У Shi et al. $\alpha=\beta=1$.

**Задание 3. Обучите DragonNet (`fit('dragonnet', alpha=1)`) и DragonNet с targeted regularization (`fit('dragonnet', alpha=1, beta=1)`) с сидами 0–4. Напишите `cate_treg(r)` — оценку эффекта по $\tilde Q$ (склонность ограничьте отрезком $[0.01, 0.99]$, как при обучении). Сведите в одну таблицу средние по сидам PEHE и ошибку среднего эффекта для TARNet, CFRNet с $\alpha=10$, DragonNet и DragonNet с targeted regularization; для последнего выведите и $\hat\varepsilon$ каждого сида. Чему равна средняя ошибка среднего эффекта у DragonNet с targeted regularization?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 5. Где ломается

### 5.1. Без ранней остановки

На 417 обучающих объектах сеть с двумя слоями по 100 нейронов легко запоминает шум наблюдаемых исходов. Ошибку прогноза исхода видно на проверочной части, а ошибку эффекта — нет: ее можно посчитать только на полусинтетических данных.

### 5.2. Разброс по сидам

Все сравнения выше — средние по пяти сидам. Сид меняет начальные веса, и с ними — найденный минимум.

**Задание 4. Обучите TARNet без ранней остановки (`fit('tarnet', early_stop=False)`, все 3000 шагов) с сидами 0–4 и выведите средний PEHE. Затем для TARNet, CFRNet с $\alpha=10$ и DragonNet выведите PEHE каждого из пяти сидов и разброс (максимум минус минимум). Что говорят эти числа о сравнении архитектур по одному сиду?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 6. Реализации

- [CATENets](https://github.com/AliciaCurth/CATENets) (лицензия BSD-3-Clause, Curth и van der Schaar): `catenets.models.torch.TARNet`, `DragonNet`, `FlexTENet` и нейросетевые мета-обучатели на `torch`, а также версии на JAX.
- [`causalml`](https://github.com/uber/causalml): `causalml.inference.tf.DragonNet` на TensorFlow и `causalml.inference.torch.CEVAE`.
- Код авторов: [cfrnet](https://github.com/clinicalml/cfrnet) (лицензия MIT, TensorFlow 1) и [dragonnet](https://github.com/claudiashi57/dragonnet) — в репозитории DragonNet нет лицензии, поэтому в тетради своя реализация.

## Что унести из тетради

- **Общее представление и две головы исхода** — TARNet — уже лучше T-learner на бустинге на этих данных, но не лучше S-learner.
- **Штраф CFRNet сближает представления групп,** и на этих данных вместе с расстоянием падают PEHE и ошибка среднего эффекта; но выигрыш в PEHE меньше разброса между сидами, а $\alpha$ на реальных данных приходится выбирать без истинного эффекта.
- **Targeted regularization работает на средний эффект,** а не на эффект отдельного объекта — она решает оценивающее уравнение для среднего.
- **Нейросеть для CATE нельзя оценивать по одному сиду и без ранней остановки:** разброс между сидами того же порядка, что разница архитектур, а переобучение видно только по ошибке эффекта, которой на реальных данных нет.

Источники:
1. [Johansson, Shalit & Sontag, 2016. Learning representations for counterfactual inference](https://arxiv.org/abs/1605.03661)
2. [Shalit, Johansson & Sontag, 2017. Estimating individual treatment effect: generalization bounds and algorithms](https://arxiv.org/abs/1606.03976)
3. [Shi, Blei & Veitch, 2019. Adapting neural networks for the estimation of treatment effects](https://arxiv.org/abs/1906.02120)
4. [Hill, 2011. Bayesian nonparametric modeling for causal inference](https://doi.org/10.1198/jcgs.2010.08162)
5. [Louizos, Shalit, Mooij & Sontag, 2017. Causal effect inference with deep latent-variable models](https://arxiv.org/abs/1705.08821); данные IHDP — первая из десяти реализаций в [репозитории CEVAE](https://github.com/AMLab-Amsterdam/CEVAE)